# Z assembly cost: `fill_mode="full_sym"` (new default) vs `"mirror"` (former default)

Step 4 (5/11). `full_sym` fills every (m, n) pair and returns (Z + Zᵀ)/2, which keeps the scalar-potential term exactly
charge-neutral on loop currents; `mirror` filled m ≤ n only. Expected cost ≈ ×2. Meshes: square loop, centre-line side 20 mm,
w = 2 mm, cosine grading, n_side = 16, n_width = 4 (N = 880) and 8 (N = 2208); one frequency, 0.9 GHz.

GPU: `impedance_matrix_z_sweep_gpu(use_gpu=True)` (the path `calculate_z_matrice` takes above 100 edges when a GPU is present).
CPU: `impedance_matrix_z`, the per-pair assembler (`calculate_z_matrice` without GPU). The CPU N = 2208 pair is long (≈ 10 + 20 min):
it only runs if `RUN_CPU_LARGE = True`, otherwise it is extrapolated in N² from N = 880.

In [1]:
import sys, time
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.optim.geometries import square_loop
from backend.utils.compute_backend import GPU_AVAILABLE, describe_backend
from backend.utils.impmet import impedance_matrix_z, impedance_matrix_z_sweep_gpu

EPS0, MU0 = 8.854e-12, 1.257e-6
C0 = 1 / np.sqrt(EPS0 * MU0)
F = 0.9e9
RUN_CPU_LARGE = False
meshes = {nw: square_loop(0.020, 0.002, 16, nw, 'cosine') for nw in (4, 8)}
print(describe_backend(), {nw: m.n_edges for nw, m in meshes.items()})

GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) {4: 880, 8: 2208}


In [2]:
def timed(fn, *a, **kw):
    t = time.perf_counter()
    out = fn(*a, **kw)
    return out, time.perf_counter() - t

results = {}
if GPU_AVAILABLE:
    w = square_loop(0.020, 0.002, 4, 1)                      # warm-up (CUDA context, kernels)
    impedance_matrix_z_sweep_gpu(w.edges, w.triangles, w.points, [F], MU0, EPS0, use_gpu=True)
    for nw, m in meshes.items():
        for mode in ('mirror', 'full_sym'):
            Z, t = timed(impedance_matrix_z_sweep_gpu, m.edges, m.triangles, m.points, [F], MU0, EPS0,
                         use_gpu=True, fill_mode=mode)
            results[('GPU', m.n_edges, mode)] = t
            print(f'GPU  N = {m.n_edges:5d}  {mode:9s} {t:8.1f} s')

GPU  N =   880  mirror        13.4 s


GPU  N =   880  full_sym      22.8 s


GPU  N =  2208  mirror        49.1 s


GPU  N =  2208  full_sym      86.8 s


In [3]:
w0 = 2 * np.pi * F
for nw, m in meshes.items():
    if m.n_edges > 1000 and not RUN_CPU_LARGE:
        continue
    for mode in ('mirror', 'full_sym'):
        Z, t = timed(impedance_matrix_z, m.edges, m.triangles, m.points, 1j * w0 / C0, w0, MU0, EPS0, fill_mode=mode)
        results[('CPU', m.n_edges, mode)] = t
        print(f'CPU  N = {m.n_edges:5d}  {mode:9s} {t:8.1f} s')

CPU  N =   880  mirror        59.0 s


CPU  N =   880  full_sym     109.7 s


In [4]:
print(f"{'backend':8s} {'N':>6s} {'mirror':>10s} {'full_sym':>10s} {'ratio':>7s}")
for dev in ('GPU', 'CPU'):
    for nw, m in meshes.items():
        a, b = results.get((dev, m.n_edges, 'mirror')), results.get((dev, m.n_edges, 'full_sym'))
        if a is None and dev == 'CPU':
            n0 = meshes[4].n_edges
            a0, b0 = results.get(('CPU', n0, 'mirror')), results.get(('CPU', n0, 'full_sym'))
            if a0 is not None:
                s = (m.n_edges / n0) ** 2
                print(f"{dev:8s} {m.n_edges:6d} {a0 * s:9.0f}s* {b0 * s:9.0f}s* {b0 / a0:7.2f}   (* extrapolated in N^2 from N = {n0})")
            continue
        if a is not None:
            print(f"{dev:8s} {m.n_edges:6d} {a:9.1f}s {b:9.1f}s {b / a:7.2f}")

backend       N     mirror   full_sym   ratio
GPU         880      13.4s      22.8s    1.70
GPU        2208      49.1s      86.8s    1.77
CPU         880      59.0s     109.7s    1.86
CPU        2208       371s*       691s*    1.86   (* extrapolated in N^2 from N = 880)
